# LSTM-arithmetic

## Dataset
- [Arithmetic dataset](https://drive.google.com/file/d/1cMuL3hF9jefka9RyF4gEBIGGeFGZYHE-/view?usp=sharing)

In [1]:
! pip install seaborn
! pip install opencc
! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
import opencc
import os
from sklearn.model_selection import train_test_split

data_path = './drive/MyDrive/NLP_HW2'
os.makedirs(data_path, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 31.3 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
import random
import pandas as pd

def make_n_digit_expression(num_digits=3, ops=('+', '-', '*')):
    """e.g. num_digits=3, (123+456)*789="""
    low = 10**(num_digits - 1)
    high = 10**num_digits - 1

    a = random.randint(low, high)
    b = random.randint(low, high)
    c = random.randint(low, high)

    op1 = random.choice(ops)
    op2 = random.choice(ops)

    # random decide the position of ()
    r = random.random()
    if r < 0.4:
        expr = f"({a}{op1}{b}){op2}{c}="
    elif r < 0.85:
        expr = f"{a}{op1}({b}{op2}{c})="
    else:
      expr = f"{a}{op1}{b}{op2}{c}="

    ans = eval(expr[:-1])

    return expr, str(ans)


def make_dataset(n_samples=50000, num_digits=3):
    data = [make_n_digit_expression(num_digits) for _ in range(n_samples)]
    return pd.DataFrame(data, columns=["src", "tgt"])


# generate 3 digit train data & 2 digit eval data
df_train = make_dataset(1200000, num_digits=3)
df_eval  = make_dataset(270000, num_digits=2)

df_train.to_csv(f"{data_path}/arithmetic_train_3digit.csv", index=False)
df_eval.to_csv(f"{data_path}/arithmetic_eval_2digit.csv", index=False)

print("Generated 3-digit training and 2-digit eval data!")


Generated 3-digit training and 2-digit eval data!


In [3]:
print(df_train.head(10))

              src      tgt
0    507+352+491=     1350
1  (871+761)*509=   830688
2  (835+924)*971=  1707989
3  (395*942)+724=   372814
4  384-(528*897)=  -473232
5  (629+766)+578=     1973
6  (475*624)+929=   297329
7    610+680*943=   641850
8  108+(237+747)=     1092
9  234*(134+309)=   103662


In [4]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [5]:
char_to_id = {}
id_to_char = {}

# write your code here
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to conver charactors to ids, while id_to_char is the opposite
chars = sorted(set("".join(df_train['src'].tolist())))
tokens = ['<pad>', '<eos>'] + chars # '<pad>': 0,'<eos>': 1
char_to_id = {ch:i for i, ch in enumerate(tokens)}
id_to_char = {i:ch for ch, i in char_to_id.items()}

vocab_size = len(char_to_id)
print('Vocab size {}'.format(vocab_size))

Vocab size 18


# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [6]:


# Write your code here
# 產生 x/y 的 id 清單。序列為：src + <eos>，x=[:-1]，y=[1:]
def encode_with_eos(s):
    ids = [char_to_id[c] for c in s] + [char_to_id['<eos>']]
    x_ids = ids[:-1]
    y_ids = ids[1:]
    # 只在 '=' 之後計算 loss：把 '=' 之前的 y 標成 <pad>
    eq_pos = s.index('=')  # 一定存在
    for t in range(eq_pos):  # y 的前 eq_pos 個位置忽略
        y_ids[t] = char_to_id['<pad>']
    return x_ids, y_ids

xy_ids = [encode_with_eos(str(s)) for s in df_train['src']]
df_train[['char_id_list', 'label_id_list']] = pd.DataFrame(xy_ids, index=df_train.index)


df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,507+352+491=1350,1350,16,"[12, 7, 14, 5, 10, 12, 9, 5, 11, 16, 8, 17, 8,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 10, 12, 7..."
1,(871+761)*509=830688,830688,20,"[2, 15, 14, 8, 5, 14, 13, 8, 3, 4, 12, 7, 16, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 10..."
2,(835+924)*971=1707989,1707989,21,"[2, 15, 10, 12, 5, 16, 9, 11, 3, 4, 16, 14, 8,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 14,..."
3,(395*942)+724=372814,372814,20,"[2, 10, 16, 12, 4, 16, 11, 9, 3, 5, 14, 9, 11,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 10, 14..."
4,384-(528*897)=-473232,-473232,21,"[10, 15, 11, 6, 2, 12, 9, 15, 4, 15, 16, 14, 3...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 11,..."


# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

In [7]:
batch_size = 64
epochs = 2
embed_dim = 256
hidden_dim = 256
lr = 0.001
grad_clip = 1

# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [8]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # return the amount of data
        return len(self.sequences) # Write your code here

    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        row = self.sequences.iloc[index]
        x = row['char_id_list'] # Write your code here
        y = row['label_id_list'] # Write your code here
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])

    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x, batch_first=True, padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y, batch_first=True, padding_value=char_to_id['<pad>'])

    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [9]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [10]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(ds_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=2) # Write your code here

# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [11]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()

        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=char_to_id['<pad>'])

        self.rnn_layer1 = torch.nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)

        self.rnn_layer2 = torch.nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)
        # self.rnn_layer1 = torch.nn.GRU(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)

        # self.rnn_layer2 = torch.nn.GRU(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)

        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim, out_features=hidden_dim), torch.nn.ReLU(), torch.nn.Linear(in_features=hidden_dim, out_features=vocab_size))

    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)

    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)

        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x, batch_x_lens, batch_first=True, enforce_sorted=False)

        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)

        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x, batch_first=True)

        batch_x = self.linear(batch_x)

        return batch_x

    def generator(self, start_char, max_len=200):

        char_list = [char_to_id[c] for c in start_char]

        next_char = None

        while len(char_list) < max_len:
            # Write your code here
            # Pack the char_list to tensor
            x = torch.tensor(char_list, dtype=torch.long, device=device).unsqueeze(0)  # [T] -> [1, T]
            lens = torch.LongTensor([x.size(1)]).to(device)
            # Input the tensor to the embedding layer, LSTM layers, linear respectively
            emb = self.embedding(x)  # [1, T, E]
            packed = torch.nn.utils.rnn.pack_padded_sequence(emb, lens, batch_first=True, enforce_sorted=False)
            h1, _ = self.rnn_layer1(packed)
            h2, _ = self.rnn_layer2(h1)
            out, _ = torch.nn.utils.rnn.pad_packed_sequence(h2, batch_first=True)
            logits = self.linear(out)
            y = logits[0, -1, :] # Obtain the next token prediction y

            next_char = int(torch.argmax(y).item()) # Use argmax function to get the next token prediction

            if next_char == char_to_id['<eos>']:
                break

            char_list.append(next_char)

        return [id_to_char[ch_id] for ch_id in char_list]

In [12]:
torch.manual_seed(2)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Write your code here. Specify a device (cuda or cpu)

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim)

In [13]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>']) # Write your code here. Cross-entropy loss function. The loss function should ignore <pad>
optimizer = torch.optim.AdamW(model.parameters(), lr=lr) # Write your code here. Use Adam or AdamW for Optimizer

# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [14]:
import re
from collections import Counter

def classify_error(inp, gt, pred):
    if '(' in inp:
        return "Parentheses"
    elif '*' in inp and ('+' in inp or '-' in inp):
        return "Mixed Operators"
    elif len(inp.split('+')) > 2 or len(inp.split('-')) > 2:
        return "Long Sequence"
    else:
        return "Simple Error"

In [15]:
from tqdm import tqdm
from copy import deepcopy
model = model.to(device)
model.train()
i = 0

for epoch in range(1, epochs+1):
    inputs, ground_truths, predictions = [], [], []
    # The process bar
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        # Write your code here
        # Clear the gradient
        optimizer.zero_grad(set_to_none=True)

        batch_pred_y = model(batch_x.to(device), batch_x_lens)

        # Write your code here
        # Input the prediction and ground truths to loss function
        # Back propagation
        loss = criterion(
            batch_pred_y.reshape(-1, batch_pred_y.size(-1)),
            batch_y.to(device).reshape(-1)
        )
        loss.backward()

        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip)

        # Write your code here
        optimizer.step()
        # Optimize parameters in the model

        i+=1
        if i%50==0:
            bar.set_postfix(loss = loss.item())

    # Evaluate your model
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        batch_x = row['src']
        batch_y = row['tgt']

        # Write your code here. Input the batch_x to the model and generate the predictions
        # Write your code here.
        pred_tokens = model.generator(batch_x)
        pred_seq = ''.join(pred_tokens)
        gen_part = pred_seq[len(batch_x):]

        # Collect wrong answer
        inputs.append(batch_x)
        ground_truths.append(batch_y)
        predictions.append(gen_part)

        # Check whether the prediction match the ground truths
        # Compute exact match (EM) on the eval dataset
        # EM = correct/total
        if gen_part == str(batch_y):
            matched += 1
        total += 1

    print(matched/total)

    # Collect wrong answer
    wrong_cases = []
    for inp, gt, pred in zip(inputs, ground_truths, predictions):
        if pred != gt:
            wrong_cases.append((inp, gt, pred))

    N = 10

    if len(wrong_cases) == 0:
        print("All predictions are correct!")
    else:
        print(f"\nFound {len(wrong_cases)} wrong generations out of {len(predictions)} samples.\n")
        print("=== Wrong Generation Examples ===")
        show_cases = wrong_cases if N is None else wrong_cases[:N]
        for i, (inp, gt, pred) in enumerate(show_cases, 1):
            print(f"{i+1}. Input: {inp}")
            print(f"Ground Truth: {gt}")
            print(f"Model Output: {pred}\n")

    # Collect why wrong
    error_types = [classify_error(inp, gt, pred) for inp, gt, pred in wrong_cases]
    counter = Counter(error_types)

    print("=== Error Type Summary ===")
    for k, v in counter.items():
        print(f"{k}: {v} cases")

    # csv
    try:
        df_wrong = pd.DataFrame(
            [(inp, gt, pred, classify_error(inp, gt, pred)) for inp, gt, pred in wrong_cases],
            columns=["input", "ground_truth", "prediction", "error_type"]
        )
        csv_path = f"{data_path}/wrong_cases_epoch{epoch}.csv"
        df_wrong.to_csv(csv_path, index=False, encoding="utf-8")
        print(f"Saved wrong cases to: {csv_path}")
    except Exception as e:
        import csv
        csv_path = f"{data_path}/wrong_cases_epoch{epoch}.csv"
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["input", "ground_truth", "prediction", "error_type"])
            for inp, gt, pred in wrong_cases:
                writer.writerow([inp, gt, pred, classify_error(inp, gt, pred)])
        print(f"Saved wrong cases to: {csv_path} (csv module)")

Train epoch 1: 100%|██████████| 18750/18750 [1:35:29<00:00,  3.27it/s, loss=1.05]
Validation epoch 1: 270000it [3:01:50, 24.75it/s]


0.00024814814814814816

Found 269933 wrong generations out of 270000 samples.

=== Wrong Generation Examples ===
2. Input: 39*64+89=
Ground Truth: 2585
Model Output: 230035

3. Input: 41*20*22=
Ground Truth: 18040
Model Output: 1888376

4. Input: 89-(78*47)=
Ground Truth: -3577
Model Output: -320035

5. Input: (62*51)*62=
Ground Truth: 196044
Model Output: 17280378

6. Input: 50-(17*84)=
Ground Truth: -1378
Model Output: -98773

7. Input: 28-41-45=
Ground Truth: -58
Model Output: -389

8. Input: 23-(46-15)=
Ground Truth: -8
Model Output: -2

9. Input: (46*53)+41=
Ground Truth: 2479
Model Output: 240035

10. Input: (67+54)+26=
Ground Truth: 147
Model Output: 1500

11. Input: (30*66)*78=
Ground Truth: 154440
Model Output: 11700380

=== Error Type Summary ===
Mixed Operators: 17981 cases
Simple Error: 13429 cases
Parentheses: 229479 cases
Long Sequence: 9044 cases
Saved wrong cases to: ./drive/MyDrive/NLP_HW2/wrong_cases_epoch1.csv


Train epoch 2: 100%|██████████| 18750/18750 [1:39:34<00:00,  3.14it/s, loss=1.05]
Validation epoch 2: 270000it [3:06:53, 24.08it/s]


0.00022592592592592591

Found 269939 wrong generations out of 270000 samples.

=== Wrong Generation Examples ===
2. Input: 39*64+89=
Ground Truth: 2585
Model Output: 238005

3. Input: 41*20*22=
Ground Truth: 18040
Model Output: 1850508

4. Input: 89-(78*47)=
Ground Truth: -3577
Model Output: -320505

5. Input: (62*51)*62=
Ground Truth: 196044
Model Output: 16595508

6. Input: 50-(17*84)=
Ground Truth: -1378
Model Output: -93002

7. Input: 28-41-45=
Ground Truth: -58
Model Output: -401

8. Input: 23-(46-15)=
Ground Truth: -8
Model Output: 22

9. Input: (46*53)+41=
Ground Truth: 2479
Model Output: 237005

10. Input: (67+54)+26=
Ground Truth: 147
Model Output: 1500

11. Input: (30*66)*78=
Ground Truth: 154440
Model Output: 11290020

=== Error Type Summary ===
Mixed Operators: 17981 cases
Simple Error: 13423 cases
Parentheses: 229492 cases
Long Sequence: 9043 cases
Saved wrong cases to: ./drive/MyDrive/NLP_HW2/wrong_cases_epoch2.csv
